# Trees, and why one is never enough

MichAl Academy, lesson 2.5.

Run each cell with **Shift+Enter**.

A decision tree is a list of yes-or-no questions that it worked out for itself.
You can print the list. That makes it the most readable model in this course,
and on its own one of the weakest, and the gap between those two facts is what
this notebook measures.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, load_digits
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)


def accuracy(model, X, y):
    return cross_val_score(model, X, y, cv=CV).mean()


## 1. Print the model

Fit a tree two questions deep to the iris measurements and ask it to write
itself out.


In [ ]:
iris = load_iris()
tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(iris.data, iris.target)

print(export_text(tree, feature_names=list(iris.feature_names), class_names=list(iris.target_names)))
print(f"leaves: {tree.get_n_leaves()}")
print(f"cross-validated accuracy: {accuracy(DecisionTreeClassifier(max_depth=2, random_state=0), iris.data, iris.target):.3f}")


That is the entire model, and it reads like a page from a field guide. Two
thresholds on one measurement, and it gets 92.7% of held-back flowers right.

Compare that with lesson 2.4's logistic regression, which needed thirty weights
and a squash. Neither is better in general. They are readable in different ways:
one gives you the strength of every measurement, the other gives you a
procedure you could follow with a ruler.


## 2. Deeper is not better

Add a level and watch two numbers move in opposite directions.


In [ ]:
print(f"{'depth':>6} {'leaves':>7} {'train':>7} {'cross-validated':>16}")
for depth in (1, 2, 3, 4, None):
    model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    cv = accuracy(model, iris.data, iris.target)
    fitted = model.fit(iris.data, iris.target)
    print(f"{str(depth):>6} {fitted.get_n_leaves():>7} {fitted.score(iris.data, iris.target):>7.3f} {cv:>16.3f}")


Depth 3 fits the training data better than depth 2 and does no better on data it
has not seen. The extra question found something that was true of these 150
flowers and not true of flowers in general.

You can see it in the printout. Look at the last split:


In [ ]:
deeper = DecisionTreeClassifier(max_depth=3, random_state=0).fit(iris.data, iris.target)
print(export_text(deeper, feature_names=list(iris.feature_names), class_names=list(iris.target_names)))


The bottom pair both predict `virginica`. The tree asked a question and then
gave the same answer either way, which is a question that cannot change a single
prediction. That is what the extra depth bought.


## 3. One tree on a harder problem

Iris is small and tidy. Try the digits from lesson 2.2, where one tree has real
work to do.


In [ ]:
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=0, stratify=digits.target
)

print(f"{'depth':>7} {'leaves':>7} {'train':>7} {'held back':>10}")
for depth in (1, 2, 3, 5, 8, 12, None):
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f"{str(depth):>7} {t.get_n_leaves():>7} {t.score(X_train, y_train):>7.3f} {t.score(X_test, y_test):>10.3f}")


An unlimited tree reaches **1.000 on the training data**, which is not learning,
it is memorising: with 136 leaves for 1,257 training images it has carved out
enough boxes to place almost every one. On images it has not seen it manages
0.843, and it stops improving after depth 8.

This is the ceiling for a single tree. The next part is how to get past it.


## 4. Why a crowd of weak trees beats one good tree

Grow five deliberately different trees. Each one gets a random sample of the
training rows, drawn with replacement, and is allowed to consider only a random
subset of the pixels at each split. Both of those make each tree *worse*.


In [ ]:
rng = np.random.default_rng(0)
predictions = []

for seed in range(5):
    rows = rng.choice(len(X_train), len(X_train), replace=True)
    t = DecisionTreeClassifier(random_state=seed, max_features="sqrt").fit(
        X_train[rows], y_train[rows]
    )
    predictions.append(t.predict(X_test))
    print(f"tree {seed}: {(predictions[-1] == y_test).mean():.3f}")

P = np.array(predictions)


Every one of them is worse than the single full tree above. Now ask how much
they disagree with each other, and then let them vote.


In [ ]:
from scipy.stats import mode

pairs = [(P[i] != P[j]).mean() for i in range(5) for j in range(i + 1, 5)]
print(f"average disagreement between two of these trees: {np.mean(pairs):.3f}")

vote = mode(P, axis=0, keepdims=False).mode
print(f"best single tree of the five: {max((p == y_test).mean() for p in P):.3f}")
print(f"majority vote of all five:    {(vote == y_test).mean():.3f}")


The vote beats every tree that went into it, by a wide margin.

The disagreement figure is the reason. These trees are wrong about different
images, so when four of them are right about a particular digit the fifth one's
mistake is outvoted. Averaging only helps when the errors are independent, which
is exactly why the recipe deliberately damages each tree: identical trees would
make identical mistakes and voting would change nothing.

That is a random forest, and now it is worth using the real one.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print(f"{'trees':>7} {'accuracy':>9}")
for n in (1, 3, 10, 30, 100, 300):
    print(f"{n:>7} {accuracy(RandomForestClassifier(n_estimators=n, random_state=0), digits.data, digits.target):>9.3f}")


One tree 0.777, ten trees 0.938, a hundred 0.973, three hundred 0.977. More
trees never makes a forest worse, and the returns flatten quickly, so a hundred
is the usual answer and tuning that number is rarely where your time goes.

The price is the thing this lesson opened with. A hundred trees cannot be
printed out, so the readability that made a single tree attractive is gone. That
loss is what lesson 2.9 exists to repair.


## 5. Support vector machines

One more method, because it turns up in inherited codebases and because the idea
behind it is worth having.

A support vector machine draws the boundary that leaves the widest possible gap
between the classes, and it can bend that boundary using a kernel, which is a
way of measuring similarity between two examples rather than working in the
original columns.

Measure it against everything above, on the same digits.


In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

candidates = [
    ("one tree",            DecisionTreeClassifier(random_state=0)),
    ("forest of 100",       RandomForestClassifier(n_estimators=100, random_state=0)),
    ("logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))),
    ("svm, linear kernel",  make_pipeline(StandardScaler(), SVC(kernel="linear"))),
    ("svm, rbf kernel",     make_pipeline(StandardScaler(), SVC())),
]
for name, model in candidates:
    print(f"{name:>21}  {accuracy(model, digits.data, digits.target):.3f}")


The support vector machine wins here, on 64 features and 1,797 samples, and it
beats both the forest and logistic regression.

Two properties explain why it was reached for so often before gradient boosting
and deep learning became the defaults. It copes well when there are many
features relative to samples, and it has no learning rate or number of rounds to
tune, so it tends to work acceptably on the first attempt. That is a large
practical advantage when compute is expensive.

What changed is not that it stopped working. It scales poorly as the number of
samples grows, because the cost grows faster than linearly, and choosing a
kernel is a judgement that gradient boosting does not ask you to make. If you
inherit a classifier from an older codebase, this is very often what it is, and
the measurement above is the reason somebody chose it.


## What to take from this

| Model | Digits accuracy | Can you read it? |
|---|---|---|
| One tree | 0.859 | Yes, print the questions |
| Forest of 100 | 0.973 | No |
| Logistic regression | 0.969 | Yes, read the weights |
| SVM, rbf kernel | 0.981 | Not usefully |

**A tree is its questions**, which makes it the only model here you can hand to
somebody with no background and have them follow it.

**Depth is where a tree goes wrong.** An unlimited tree hit 1.000 on training
data and 0.843 on held-back data. Lesson 2.11 is about that gap.

**Averaging works because the members disagree.** Five trees that were each
worse than a single tree, and that disagreed with each other 38% of the time,
voted their way past all of them.

**Every step up in accuracy so far has cost readability.** That trade is the
subject of lesson 2.9.
